In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from torch.utils.data import  TensorDataset, DataLoader
from sklearn.preprocessing import  StandardScaler
from sklearn.model_selection import  train_test_split


In [2]:
df = pd.read_csv(r"C:\Users\pc\OneDrive\Music\Desktop\ML_Projects\Dataset\powerplant_data.csv")
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9568 entries, 0 to 9567
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      9568 non-null   float64
 1   V       9568 non-null   float64
 2   AP      9568 non-null   float64
 3   RH      9568 non-null   float64
 4   PE      9568 non-null   float64
dtypes: float64(5)
memory usage: 373.9 KB


In [4]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [7]:
# Feature x y split

x = df.drop(["PE"], axis=1)
y = df["PE"]

# Train Test Split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [8]:
# Feature Scaling

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)


In [9]:
# Comvert dataset into tensor dataset

x_train_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

x_test_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [11]:
# Tensor Dataset & Dataloader

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [12]:
# ANN Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(x_train.shape[1], 6),
            nn.ReLU(),

            nn.Linear(6, 6),
            nn.ReLU(),

            nn.Linear(6, 1),
        )

    def forward(self, x):
        return self.model(x)

In [13]:
# Loss & Optimizer

import torch.optim as optim

model = ANN()

crietion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [14]:
# Train the ANN Neural

train_losses = []
val_losses = []

best_val_loss = float("inf")

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # Total traing loss for 1 epoch

    for xb, yb, in train_loader:
        # xb = feature of 1 batch
        # yb = label
        optimizer.zero_grad()

        outputs = model(xb) # forward prop....predicted
        loss = crietion(outputs, yb) # compute loss
        loss.backward() # back prop..compute graditents
        optimizer.step() # param update

        running_loss += loss.item() # loss is a tensor
    
    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


# Validition 
model.eval()
running_val_loss = 0.0

with torch.no_grad():   # no gradients compute
    for xb, yb in test_loader:
        outputs = model(xb)
        loss = crietion(outputs, yb)
        running_val_loss += loss 

epoch_val_loss = running_val_loss   / len(test_loader)
val_losses.append(epoch_val_loss)

print(f"epoch{epoch+1}/{epochs}==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

if epoch_val_loss < best_val_loss:
    beest_val_loss = epoch_val_loss
    torch.save(model.state_dict(), "best_model.pt") # pt or .pt

epoch100/100==> train loss = 21.40448767741521 & val loss = 19.994626998901367


In [16]:
# Loading the best model|
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [17]:
# Evalutiion

model.eval()
with torch.no_grad():
    train_preds = model(x_train_tensor)
    test_preds = model(x_test_tensor)

    train_mse_loss  = crietion(train_preds, y_train_tensor)
    test_mse_loss = crietion(test_preds, y_test_tensor)

print("Training MSE", train_mse_loss.item())
print("Testing MSE", test_mse_loss.item())

Training MSE 21.3211669921875
Testing MSE 20.00605583190918


In [18]:
# R2 Score Metric Accuracy

from sklearn.metrics import  r2_score

r2 = r2_score(y_test, test_preds)*100
r2

93.00840007902593

In [20]:
# Accutal Value & Model Prediction Value

predicted_df = pd.DataFrame(test_preds.numpy(), columns=["Prediction Values"])
actual_df = pd.DataFrame(y_test.values, columns=["Actual Values"])

pd.concat([predicted_df, actual_df], axis=1)

,Prediction Values,Actual Values
0,436.606873,433.27
1,437.819733,438.16
2,460.995880,458.42
3,475.297241,480.82
4,436.545166,441.41
...,...,...
1909,451.528900,456.70
1910,432.819031,438.04
1911,467.847717,467.80
1912,432.299255,437.14
